In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b")


In [3]:
from pydantic import BaseModel,Field
from typing import Literal
class Task(BaseModel):
  task_id:str
  task_title:str
  department:Literal["WEB","SEO","MARKETTING","QA","AI"]
  required_role:str
  priority:Literal["HIGH","MEDIUM","LOW"]
  depends_on:list[str]=Field(default_factory=list)
  acceptance_criteria:str
class ProjectPlan(BaseModel):
  project_name:set
  tasks:list[Task]


In [4]:
planner=model.with_structured_output(ProjectPlan)

In [5]:
from langchain_core.prompts import PromptTemplate
planning_prompt=PromptTemplate(
    template="""
    You are an office project planner.

Break the manager's project brief into actionable tasks.

Rules:
- Use only the allowed departments and priorities.
- Assign a suitable employee role to each task.
- Give every task a unique ID, such as T1 or T2.
- Use depends_on for tasks that must finish first.
- Include clear acceptance criteria.
- Do not invent employee names.
- Make QA testing depend on the relevant development tasks.

Project brief:
{brief}
    """,
    input_variables=["brief"]
)

In [6]:
plan =planner.invoke(
    planning_prompt.format(brief="""Build a company website with a homepage, SEO optimization,
and testing before launch.""")
)

In [7]:
print(plan.model_dump_json(indent=2))

{
  "project_name": [
    "Company Website Launch"
  ],
  "tasks": [
    {
      "task_id": "T1",
      "task_title": "Design Homepage Layout",
      "department": "WEB",
      "required_role": "UI/UX Designer",
      "priority": "HIGH",
      "depends_on": [],
      "acceptance_criteria": "Wireframe and high‑fidelity mockup approved by stakeholders."
    },
    {
      "task_id": "T2",
      "task_title": "Implement Homepage",
      "department": "WEB",
      "required_role": "Front‑end Developer",
      "priority": "HIGH",
      "depends_on": [
        "T1"
      ],
      "acceptance_criteria": "Responsive homepage built, passes visual regression tests and matches approved design."
    },
    {
      "task_id": "T3",
      "task_title": "Implement SEO Metadata",
      "department": "SEO",
      "required_role": "SEO Specialist",
      "priority": "MEDIUM",
      "depends_on": [
        "T1"
      ],
      "acceptance_criteria": "Meta tags, titles, descriptions, canonical URLs, and st

In [8]:
employees=[
    {
        "employee_id":"E1",
        "name":"Jawad",
        "role":"Frontend Developer",
        "department":"WEB",
        "workload":0
    },
    {
        "employee_id":"E2",
        "name":"Ali",
        "role":"SEO Specialist",
        "department":"SEO",
        "workload":0
    },
    {
        "employee_id":"E3",
        "name":"Khurram",
        "role":"AI Engineer",
        "department":"AI",
        "workload":0
    },
    {
        "employee_id":"E4",
        "name":"Hanan",
        "role":"Marketting Executive",
        "department":"MARKETTING",
        "workload":0
    },
     {
        "employee_id":"E5",
        "name":"Hasaan",
        "role":"QA",
        "department":"QA",
        "workload":0
    },
     {
        "employee_id":"E6",
        "name":"Asim",
        "role":"Backend Engineer",
        "department":"WEB",
        "workload":0
    }

]

tasks={}

In [9]:
tasks

{}

In [10]:
def validate_plan(plan,employees):
    task_ids=[task.task_id for task in plan.tasks]
    #the task ids must be unique
    if(len(task_ids)!=len(set(task_ids))):
      raise ValueError("Duplicate TAsk IDs found")
    valid_roles={employee["role"] for employee in employees}
    valid_ids=set(task_ids)

    for task in plan.tasks:
      if task.task_id not in valid_ids:
        raise ValueError(f"NO employee role available for task {task.task_title}")
      for dependency in  task.depends_on:
        if dependency not in valid_ids:
          raise ValueError(f"{task.task_id} depends on unknown task named as :{dependency}")
      if task.task_id in task.depends_on:
          raise ValueError("A task cannot be dependent on itself")
    return True



In [11]:
validate_plan(plan,employees)
print("plan passed the validation")

plan passed the validation


{
  "project_name": [
    "Company Website Project"
  ],
  "tasks": [
    {
      "task_id": "T1",
      "task_title": "Develop Homepage",
      "deparmant": "WEB",
      "required_role": "Web Developer",
      "priority": "high",
      "depends_on": "none",
      "acceptance_criteria": "Homepage is fully responsive and deployed to the staging environment."
    },
    {
      "task_id": "T2",
      "task_title": "Implement SEO Optimization",
      "deparmant": "SEO",
      "required_role": "SEO Specialist",
      "priority": "medium",
      "depends_on": "T1",
      "acceptance_criteria": "Meta tags, page titles, and image alt attributes are added for search engine indexing."
    },
    {
      "task_id": "T3",
      "task_title": "QA Testing and Bug Fixing",
      "deparmant": "QA",
      "required_role": "QA Tester",
      "priority": "high",
      "depends_on": "T1",
      "acceptance_criteria": "All site functionalities are tested across major browsers with no critical bugs remaini

In [13]:
tasks={}

In [14]:
print("Available Task IDs:")

for task in plan.tasks:
    print(f"\nTask: {task.task_title}")
    print(f"Task ID: {task.task_id}")
    print(f"Dependencies: {task.depends_on}")

Available Task IDs:

Task: Design Homepage Layout
Task ID: T1
Dependencies: []

Task: Implement Homepage
Task ID: T2
Dependencies: ['T1']

Task: Implement SEO Metadata
Task ID: T3
Dependencies: ['T1']

Task: Optimize Images & Performance
Task ID: T4
Dependencies: ['T2']

Task: Implement Analytics Tracking
Task ID: T5
Dependencies: ['T2']

Task: QA Testing
Task ID: T6
Dependencies: ['T2', 'T3', 'T4', 'T5']

Task: Launch Website
Task ID: T7
Dependencies: ['T6']


In [18]:
for task in plan.tasks:
        task.task_id={
        **task.model_dump(),
        "assigned_to":None,
        "Status":"TODO",
        "completion_theory":None
    }

## Find Employees By Role

In [20]:
from langchain_core.tools import tool
@tool
def get_employees_by_role(role:str)->list[dict]:
    """find employees who have requested job role"""
    return[
        {
            "employee_id":employees["employee_id"],
            "name":employees["name"],
            "role":employees["role"],
            "department":employees["department"],
            "workload":employees["workload"]

        }

        for employee in employees
        if employee["role"].lower()==role.lower
    
        

        
    ]

In [23]:
get_employees_by_role.invoke({"role":"Frontend Developer"})

[]

## Assign a task

In [24]:
@tool
def assign_a_task(task_id:str,employee_id:str)->str:
    """assign a task to a matching employee"""
    if task_id not in tasks:
        return f"Error: task{task.task_id} not in tasks"
    employee=next(
        {
            e for e in employees
            if e["employee_id"]==employee_id
        },
        None
    )
    if employee is None:
        return f"error : {employee} doesnt exists"
    task=tasks[task_id]
    if task ["assigned to"] is not None:
        return f"task has also been asssigned"
    if employee["role"] !=task["required_role"]:
        return "Error: eployee role doent matches the task"
    if employee["department"] != task["department"]:
        return "Error : employee department doesnt matches the task department"
    task["assigned_to"]=employee_id
    employee["workload"]+=1

## Employee TODO List

In [25]:
@tool
def get_employees_todo_list(employee_id:str)->list[dict]:
    """get all the tasks assigned to a specific employee."""
    return[
        {
            "task_id":task["task_id"],
            "title":task["task_title"],
            "priority":task["priority"],
            "status":task["status"]
        }
        for task_id,task in task.items()
        if task["assigned_to"]==employee_id
    ]

## update task status

In [26]:
@tool
def update_task_status(task_id:str,newstatus)->str:
    """update a task to a new status"""
    allowed_statuses=["TODO","BLOCKED","INPROGRESS","INREVIEW"]
    if task_id not in tasks:
        return f"Error: {task_id} does not exists"
    if newstatus not in allowed_statuses:
        return f"Error: {newstatus} is invalid "
    tasks[task_id]["status"]=newstatus

    return f"task {task_id} has updated to a new status {newstatus}"


In [27]:
tools=[get_employees_by_role,assign_a_task,get_employees_todo_list,update_task_status]
tools_by_name={
    tool.name:tool
    for tool in tools
}
llm_with_tools=model.bind_tools(tools)

In [34]:
from langchain_core.messages import HumanMessage,AIMessage,ToolMessage
messages=[
    # HumanMessage(content="""
    #    Assign every currently unassigned task to an existing employee
    #    with the exact required role and department.

    #    Use the available tools to find employees and assign tasks.
    #    Never invent employee IDs. Report any task you cannot assign.
    # """)
      HumanMessage(content="""
           We got a project will be having CRM (web application).assign the tasks accordingly to the team and also update the project plan 
           and make sure to use the tools that are binded to the model 
        """)

]
response=llm_with_tools.invoke(messages)



In [35]:
print("Model Response :",response.content)

Model Response : I’d love to get a bit more detail so I can make the assignments accurate:

1. **Team Composition** – Which roles are available on the team (e.g., Front‑End Developer, Back‑End Developer, UI/UX Designer, QA Engineer, Project Manager, etc.) and the employee IDs for each?  
2. **Task List** – A list of the specific tasks you’d like to assign, along with any existing task IDs (or let me know if you’d like me to create placeholder IDs).  
3. **Priority / Dependencies** – Any tasks that must be completed before others, or priority levels you’d like reflected in the plan.

Once I have that, I can:

- Fetch the relevant employees via `get_employees_by_role`.
- Assign each task to the appropriate employee with `assign_a_task`.
- Update the project plan (e.g., a high‑level timeline or Gantt‑style overview).

Let me know the details, and I’ll take it from there!


In [36]:
print("Requested Tool Calls",response.tool_calls)

Requested Tool Calls []


In [37]:
messages.append(response)

In [38]:
messages

[HumanMessage(content='\n           We got a project will be having CRM (web application).assign the tasks accordingly to the team and also update the project plan \n           and make sure to use the tools that are binded to the model \n        ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='I’d love to get a bit more detail so I can make the assignments accurate:\n\n1. **Team Composition** – Which roles are available on the team (e.g., Front‑End Developer, Back‑End Developer, UI/UX Designer, QA Engineer, Project Manager, etc.) and the employee IDs for each?  \n2. **Task List** – A list of the specific tasks you’d like to assign, along with any existing task IDs (or let me know if you’d like me to create placeholder IDs).  \n3. **Priority / Dependencies** – Any tasks that must be completed before others, or priority levels you’d like reflected in the plan.\n\nOnce I have that, I can:\n\n- Fetch the relevant employees via `get_employees_by_role`.\n- Assign each tas

In [39]:
for call in response.tool_calls:
    tool_name=call["name"]
    tool_args=call["args"]

    print(f"Executing tool{tool_name}")
    print(f"executing tool arguments :{tool_args}")


    result= tools_by_name[tool_name].invoke(tool_args)
    print("Result :",result)


    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )
    )

In [40]:
final_response=llm_with_tools.invoke(messages)

In [41]:
print(final_response)

content='Got it—let’s get the assignments rolling!  \nTo make sure I match the right people to the right work and keep the project plan up‑to‑date, I’ll need a few details first:\n\n| What I need | Why it matters | How you can provide it |\n|-------------|----------------|------------------------|\n| **Team roster** – a list of all employees who will be working on the CRM, with their roles (e.g., Front‑End Dev, Back‑End Dev, UI/UX Designer, QA, PM, etc.) and their employee IDs. | The tools need to know *who* is available for each role. | Paste a table or list, e.g. `{"role":"Front‑End Dev","employee_id":"E123"}` |\n| **Task list** – the specific tasks you want to assign (e.g., “Design login page”, “Implement OAuth”, “Write unit tests for API”). If you already have task IDs, great; otherwise I can create placeholder IDs. | I’ll use these to create assignments via `assign_a_task`. | Provide a numbered list or JSON array of `{task_id, task_name}` |\n| **Priority / dependencies** – any tas

In [42]:
final_response.tool_calls

[]

In [43]:
final_response.content

'Got it—let’s get the assignments rolling!  \nTo make sure I match the right people to the right work and keep the project plan up‑to‑date, I’ll need a few details first:\n\n| What I need | Why it matters | How you can provide it |\n|-------------|----------------|------------------------|\n| **Team roster** – a list of all employees who will be working on the CRM, with their roles (e.g., Front‑End Dev, Back‑End Dev, UI/UX Designer, QA, PM, etc.) and their employee IDs. | The tools need to know *who* is available for each role. | Paste a table or list, e.g. `{"role":"Front‑End Dev","employee_id":"E123"}` |\n| **Task list** – the specific tasks you want to assign (e.g., “Design login page”, “Implement OAuth”, “Write unit tests for API”). If you already have task IDs, great; otherwise I can create placeholder IDs. | I’ll use these to create assignments via `assign_a_task`. | Provide a numbered list or JSON array of `{task_id, task_name}` |\n| **Priority / dependencies** – any tasks that 